In [1]:
# Core
import os
import importlib
import numpy as np
import pandas as pd
from collections import Counter

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Sampling
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Model / Feature selection / Scaling
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import RFE
from sklearn.svm import SVC, LinearSVC

# Metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

# Saving Model
import joblib


from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform
# Metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

# Saving Model
import joblib

from scipy.stats import loguniform, randint


In [3]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

BASE_DIR = Path(".")

model_files = {
    "svm": "svm.pkl",
    "svm_irus": "svm_irus.pkl",
    "svm_irus_Norway": "svm_irus_Norway.pkl",
    "svm_irus_rfe": "svm_irus_rfe.pkl",
    "svm_irus_rfe_Norway": "svm_irus_rfe_Norway.pkl",
    "svm_Norway": "svm_Norway.pkl",
    "svm_rfe": "svm_rfe.pkl",
    "svm_rfe_Norway": "svm_rfe_Norway.pkl",
    "svm_smote": "svm_smote.pkl",
    "svm_smote_Norway": "svm_smote_Norway.pkl",
    "svm_smote_rfe": "svm_smote_rfe.pkl",
    "svm_smote_rfe_Norway": "svm_smote_rfe_Norway.pkl",

    "svm_purified": "Purified/svm.pkl",
    "svm_irus_purified": "Purified/svm_irus.pkl",
    "svm_irus_Norway_purified": "Purified/svm_irus_Norway.pkl",
    "svm_irus_rfe_purified": "Purified/svm_irus_rfe.pkl",
    "svm_irus_rfe_Norway_purified": "Purified/svm_irus_rfe_Norway.pkl",
    "svm_Norway_purified": "Purified/svm_Norway.pkl",
    "svm_rfe_purified": "Purified/svm_rfe.pkl",
    "svm_rfe_Norway_purified": "Purified/svm_rfe_Norway.pkl",
    "svm_smote_purified": "Purified/svm_smote.pkl",
    "svm_smote_Norway_purified": "Purified/svm_smote_Norway.pkl",
    "svm_smote_rfe_purified": "Purified/svm_smote_rfe.pkl",
    "svm_smote_rfe_Norway_purified": "Purified/svm_smote_rfe_Norway.pkl",
}

def to_serializable(obj):
    if obj is None:
        return None

    if isinstance(obj, (str, int, float, bool)):
        return obj

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        return float(obj)

    if isinstance(obj, (np.bool_,)):
        return bool(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, tuple):
        return [to_serializable(x) for x in obj]

    if isinstance(obj, list):
        return [to_serializable(x) for x in obj]

    if isinstance(obj, dict):
        return {str(k): to_serializable(v) for k, v in obj.items()}

    # sklearn estimators / transformers / pipelines
    if hasattr(obj, "get_params"):
        return repr(obj)

    return str(obj)

def shape_or_none(x):
    return tuple(x.shape) if hasattr(x, "shape") else None

summary_rows = []
all_params = {}

for model_name, rel_path in model_files.items():
    full_path = BASE_DIR / rel_path

    if not full_path.exists():
        print(f"Missing file: {full_path}")
        continue

    model = joblib.load(full_path)
    file_size_bytes = full_path.stat().st_size
    file_size_mb = file_size_bytes / (1024 ** 2)

    has_named_steps = hasattr(model, "named_steps")
    step_names = list(model.named_steps.keys()) if has_named_steps else []

    svm_step = model.named_steps["svm"] if has_named_steps and "svm" in model.named_steps else model
    rfe_step = model.named_steps["rfe"] if has_named_steps and "rfe" in model.named_steps else None

    support_vectors = getattr(svm_step, "support_vectors_", None)
    dual_coef = getattr(svm_step, "dual_coef_", None)
    coef = getattr(svm_step, "coef_", None) if hasattr(svm_step, "coef_") else None
    intercept = getattr(svm_step, "intercept_", None)
    n_support = getattr(svm_step, "n_support_", None)
    classes = getattr(svm_step, "classes_", None)

    summary_rows.append({
        "model_name": model_name,
        "file_path": str(full_path),
        "file_size_bytes": int(file_size_bytes),
        "file_size_mb": round(file_size_mb, 2),
        "object_type": type(model).__name__,
        "steps": " -> ".join(step_names) if step_names else None,

        "svm_type": type(svm_step).__name__,
        "kernel": getattr(svm_step, "kernel", None),
        "C": getattr(svm_step, "C", None),
        "gamma": getattr(svm_step, "gamma", None),
        "degree": getattr(svm_step, "degree", None),
        "coef0": getattr(svm_step, "coef0", None),
        "probability": getattr(svm_step, "probability", None),
        "class_weight": str(getattr(svm_step, "class_weight", None)),
        "decision_function_shape": getattr(svm_step, "decision_function_shape", None),
        "random_state": getattr(svm_step, "random_state", None),

        "classes": str(classes.tolist() if hasattr(classes, "tolist") else classes),
        "n_classes": len(classes) if classes is not None else None,
        "n_support_total": int(np.sum(n_support)) if n_support is not None else None,
        "n_support_per_class": str(n_support.tolist() if hasattr(n_support, "tolist") else n_support),
        "support_vectors_shape": str(shape_or_none(support_vectors)),
        "dual_coef_shape": str(shape_or_none(dual_coef)),
        "coef_shape": str(shape_or_none(coef)),
        "intercept_shape": str(shape_or_none(intercept)),

        "rfe_present": rfe_step is not None,
        "rfe_n_features_selected": getattr(rfe_step, "n_features_", None) if rfe_step is not None else None,
        "rfe_support_count": int(np.sum(rfe_step.support_)) if rfe_step is not None and hasattr(rfe_step, "support_") else None,
        "rfe_step_size": getattr(rfe_step, "step", None) if rfe_step is not None else None,
    })

    all_params[model_name] = {
        "file_path": str(full_path),
        "file_size_bytes": int(file_size_bytes),
        "file_size_mb": round(file_size_mb, 2),
        "model_type": type(model).__name__,
        "pipeline_steps": step_names,
        "pipeline_params": to_serializable(model.get_params(deep=True)),
        "svm_params": to_serializable(svm_step.get_params(deep=True)),
        "trained_attributes": {
            "classes_": to_serializable(classes),
            "n_support_": to_serializable(n_support),
            "support_vectors_shape": to_serializable(shape_or_none(support_vectors)),
            "dual_coef_shape": to_serializable(shape_or_none(dual_coef)),
            "coef_shape": to_serializable(shape_or_none(coef)),
            "intercept_shape": to_serializable(shape_or_none(intercept)),
        }
    }

    if rfe_step is not None:
        all_params[model_name]["rfe_params"] = to_serializable(rfe_step.get_params(deep=True))
        all_params[model_name]["rfe_trained_attributes"] = {
            "n_features_": to_serializable(getattr(rfe_step, "n_features_", None)),
            "support_sum": to_serializable(int(np.sum(rfe_step.support_)) if hasattr(rfe_step, "support_") else None),
            "ranking_shape": to_serializable(shape_or_none(getattr(rfe_step, "ranking_", None))),
        }

summary_df = pd.DataFrame(summary_rows).sort_values("model_name").reset_index(drop=True)

pd.set_option("display.max_columns", None)
display(summary_df)

summary_df.to_csv(BASE_DIR / "svm_model_summary.csv", index=False)

with open(BASE_DIR / "svm_model_params.json", "w", encoding="utf-8") as f:
    json.dump(all_params, f, indent=2, ensure_ascii=False)

print("\nSaved:")
print(BASE_DIR / "svm_model_summary.csv")
print(BASE_DIR / "svm_model_params.json")

print("\nExample: SVM params for one model")
print(json.dumps(all_params["svm_smote_rfe_Norway"]["svm_params"], indent=2, ensure_ascii=False))


c:\Users\Public\Anaconda\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Public\Anaconda\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Public\Anaconda\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RFE from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. U

,model_name,file_path,file_size_bytes,file_size_mb,object_type,steps,svm_type,kernel,C,gamma,degree,coef0,probability,class_weight,decision_function_shape,random_state,classes,n_classes,n_support_total,n_support_per_class,support_vectors_shape,dual_coef_shape,coef_shape,intercept_shape,rfe_present,rfe_n_features_selected,rfe_support_count,rfe_step_size
0,svm,svm.pkl,54612932,52.08,Pipeline,svm,SVC,poly,0.002781,0.037566,2,0.027156,True,None,ovr,42,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",5,352,"[35, 24, 163, 97, 33]","(352, 19277)","(4, 352)",None,"(10,)",False,NaN,NaN,NaN
1,svm_Norway,svm_Norway.pkl,33273204,31.73,Pipeline,svm,SVC,poly,0.002781,0.037566,2,0.027156,True,None,ovr,42,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",5,198,"[23, 30, 87, 48, 10]","(198, 20789)","(4, 198)",None,"(10,)",False,NaN,NaN,NaN
2,svm_Norway_purified,Purified\svm_Norway.pkl,34300932,32.71,Pipeline,svm,SVC,poly,0.473147,0.021423,4,0.002511,True,None,ovr,42,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",5,202,"[22, 30, 86, 50, 14]","(202, 21101)","(4, 202)",None,"(10,)",False,NaN,NaN,NaN
3,svm_irus,svm_irus.pkl,36712964,35.01,Pipeline,irus -> svm,SVC,linear,1.408147,scale,3,0.000000,True,None,ovr,42,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",5,234,"[34, 23, 69, 80, 28]","(234, 19277)","(4, 234)","(10, 19277)","(10,)",False,NaN,NaN,NaN
4,svm_irus_Norway,svm_irus_Norway.pkl,23453060,22.37,Pipeline,irus -> svm,SVC,linear,0.007007,scale,3,0.000000,True,None,ovr,42,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",5,137,"[24, 30, 30, 43, 10]","(137, 20789)","(4, 137)","(10, 20789)","(10,)",False,NaN,NaN,NaN
5,svm_irus_Norway_purified,Purified\svm_irus_Norway.pkl,25031076,23.87,Pipeline,irus -> svm,SVC,rbf,595.228572,0.000018,3,0.000000,True,None,ovr,42,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",5,146,"[21, 30, 35, 46, 14]","(146, 21101)","(4, 146)",None,"(10,)",False,NaN,NaN,NaN
6,svm_irus_purified,Purified\svm_irus.pkl,37370052,35.64,Pipeline,irus -> svm,SVC,rbf,199.379982,0.000042,3,0.000000,True,None,ovr,42,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",5,240,"[33, 23, 72, 83, 29]","(240, 19277)","(4, 240)",None,"(10,)",False,NaN,NaN,NaN
7,svm_irus_rfe,svm_irus_rfe.pkl,2482541,2.37,Pipeline,irus -> rfe -> svm,SVC,linear,14.747201,scale,3,0.000000,True,None,ovr,42,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",5,215,"[28, 23, 66, 75, 23]","(215, 1000)","(4, 215)","(10, 1000)","(10,)",True,1000.0,1000.0,0.50
8,svm_irus_rfe_Norway,svm_irus_rfe_Norway.pkl,1917341,1.83,Pipeline,irus -> rfe -> svm,SVC,poly,14.952245,0.012193,5,0.008207,True,None,ovr,42,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",5,138,"[23, 30, 30, 45, 10]","(138, 1000)","(4, 138)",None,"(10,)",True,1000.0,1000.0,0.25
9,svm_irus_rfe_Norway_purified,Purified\svm_irus_rfe_Norway.pkl,1880813,1.79,Pipeline,irus -> rfe -> svm,SVC,rbf,3.315179,0.067031,3,0.000000,True,None,ovr,42,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",5,158,"[27, 30, 36, 51, 14]","(158, 1000)","(4, 158)",None,"(10,)",True,1000.0,1000.0,0.20



Saved:
svm_model_summary.csv
svm_model_params.json

Example: SVM params for one model
{
  "C": 7.115245297181689,
  "break_ties": false,
  "cache_size": 200,
  "class_weight": null,
  "coef0": 0.0021705003488711127,
  "decision_function_shape": "ovr",
  "degree": 5,
  "gamma": 0.008447014808943042,
  "kernel": "poly",
  "max_iter": -1,
  "probability": true,
  "random_state": 42,
  "shrinking": true,
  "tol": 0.001,
  "verbose": false
}
